## Computational Physics -- Problem Set 4
##### _Theodore Landa_

In [ ]:
import numpy as np
from numpy.polynomial import Polynomial
import pandas as pd
import pandas as pd
import math
import time
import matplotlib.pyplot as plt
import scipy.integrate as sp
import pickle
from functools import reduce
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
from scipy.constants import Boltzmann
from scipy.constants import hbar
from scipy.constants import c
from scipy.linalg import lu_factor, lu_solve

### Problem 1.
_(a)_ We get `13.5`.

_(b)_ We get `13.699583137042454`.

_(c)_ Everything's cool!!!!

In [ ]:
fn = 'ps04_peak.pkl'
xs, ys = pickle.load(open(fn, 'rb'))

In [ ]:
print(xs, ys)

In [ ]:
maxind = np.argmax(ys)
print(ys[maxind])
print(xs[maxind])

In [ ]:
lnpeakys = np.log(ys[[maxind - 1, maxind, maxind + 1]])
peakxs = xs[[maxind - 1, maxind, maxind + 1]]
lnpoly = Polynomial.fit(peakxs, lnpeakys, 2)
lnpolyargmax = -1*lnpoly.convert().coef[1] / (2*lnpoly.convert().coef[2]) #convert() bullshit why is numpy so adversarial
print(lnpolyargmax)

In [ ]:
xpoly = np.linspace(0, 20, 10000)
ypoly = np.exp(lnpoly(xpoly))

In [ ]:
plt.title("Pickle file data and fit")
plt.scatter(xs, ys, color="k", label = "data")
plt.plot(xpoly, ypoly, label = "log space parabola (aka Gaussian)")
plt.axvline(lnpolyargmax, color="pink", label = "argmax of parabola")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()

### Problem 2.
_(a)_ We wish to evaluate $$W=\int_0^{\infty}I(\omega)d\omega=\frac{\hbar}{4\pi^2c^2}\int_0^{\infty}\frac{\omega^3}{e^{\hbar\omega/K_{B}T}-1}d\omega,$$ so we change variables as $x=\frac{\hbar\omega}{K_{B}T}$, $dx=\frac{\hbar}{K_{B}T}d\omega$. Then, rewriting the integral, we have $$W=\int_0^{\infty}I(\omega)d\omega=\frac{\hbar}{4\pi^2c^2}\int_0^{\infty}\frac{\omega^3}{e^{\hbar\omega/K_{B}T}-1}d\omega=\frac{K_{B}^4T^4}{4\pi^2c^2\hbar^3}\int_0^{\infty}\frac{x^3}{e^x-1}dx$$

_(b)_ I used `scipy.integrate.quad` for the integral in the above expression, and I got `6.49393940226683` with an upper bound on the error at `2.628470915310923e-09`, which is pretty small! So I'm fairly confident. (Note that this question asks just for the integral in the expression, which is unitless, because $x$ is unitless the way we defined it!)

_(c)_ My code gets `5.67037441918443e-08` W m^-2 K^-4, which matches all the significant figures Wikipedia lists (and goes beyond that)!! To 3 significant figures, this is `5.67` W m^-2 K^-4.

In [ ]:
def foo(x):
    return x**3 / (np.exp(x)-1)

In [ ]:
integ = sp.quad(foo, 0., np.inf)
print(integ)

In [ ]:
print(integ[0]*Boltzmann**4 / (4*np.pi**2*c**2*hbar**3))

### Problem 3.

_(a)_ For our first (most positive) eigenvalue, we get `9.000000`.

_(b)_ For our most negative, we get `-3.000000`.

_(c)_ For our smallest in magnitude, we get `-1.000000`.

_(d)_ For our closest to 4, we get `3.000000`

In [ ]:
eigenA = np.array([[0, 1, 2, 2, 1],
                  [1, 0, 0, 2, 3],
                  [2, 0, 3, 4, 0],
                  [2, 2, 4, 3, 2],
                  [1, 3, 0, 2, 0]])

In [ ]:
def random_vector(d):
    rng = np.random.default_rng()
    return rng.random(d)

def power_method(A, tol):
    if A.shape[0] != A.shape[1]:
        raise ValueError("A must be a square matrix.")

    # Choose a random initial vector to reduce the chance
    # that it is orthogonal to the dominant eigenvector.
    b_k = random_vector(A.shape[1])

    # Normalize the initial vector.
    b_k /= np.linalg.norm(b_k)

    # Multiply by the matrix.
    b_k1 = A @ b_k

    # Compute the length of the new vector.
    b_k1_norm = np.linalg.norm(b_k1)
        
    # Normalize the vector for the next iteration.
    b_k = b_k1 / b_k1_norm
    
    compare = 0.

    while (np.abs(compare-b_k1_norm) > tol):
        # Multiply by the matrix.
        compare = b_k1_norm
        
        b_k1 = A @ b_k

        # Compute the length of the new vector.
        b_k1_norm = np.linalg.norm(b_k1)

        # Normalize the vector for the next iteration.
        b_k = b_k1 / b_k1_norm

    # Return the approximate dominant eigenvector.
    return b_k1_norm

In [ ]:
print(power_method(eigenA, 5e-7))

In [ ]:
def power_method(A, tol=1e-10, max_iter=10_000):
    """Dominant (largest |λ|) eigenpair, with the sign of λ preserved."""
    A = np.asarray(A, dtype=float)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A must be a square matrix.")

    b = np.random.randn(A.shape[0])
    b /= np.linalg.norm(b)

    lam_old = 0.0
    for _ in range(max_iter):
        w = A @ b
        lam = b @ w                      # Rayleigh quotient (b is unit length)
        norm_w = np.linalg.norm(w)
        if norm_w == 0:                  # b landed in the null space
            return 0.0, b
        b = w / norm_w
        if abs(lam - lam_old) < tol * max(1.0, abs(lam)):
            return lam, b
        lam_old = lam
    raise RuntimeError("Power method did not converge.")

In [ ]:
def shifted_inverse_iteration(A, sigma=0.0, tol=1e-10, max_iter=10_000):
    """Eigenpair with eigenvalue closest to sigma.
    sigma=0 gives the smallest-magnitude eigenvalue (plain inverse iteration)."""
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    if A.ndim != 2 or n != A.shape[1]:
        raise ValueError("A must be a square matrix.")

    lu = lu_factor(A - sigma * np.eye(n))   # factor once, reuse every iteration

    b = np.random.randn(n)
    b /= np.linalg.norm(b)

    lam_old = np.inf
    for _ in range(max_iter):
        w = lu_solve(lu, b)                 # solves (A - σI) w = b, no explicit inverse
        b = w / np.linalg.norm(w)
        lam = b @ A @ b                     # Rayleigh quotient with the ORIGINAL A
        if abs(lam - lam_old) < tol * max(1.0, abs(lam)):
            return lam, b
        lam_old = lam
    raise RuntimeError("Inverse iteration did not converge.")

In [ ]:
def extreme_eigenvalues(A, **kw):
    """Most positive and most negative eigenvalues (real spectrum assumed)."""
    A = np.asarray(A, dtype=float)
    lam_dom, _ = power_method(A, **kw)
    # Shift so every eigenvalue has the same sign; the dominant one of the
    # shifted matrix is then the opposite end of the spectrum.
    mu, _ = power_method(A - lam_dom * np.eye(A.shape[0]), **kw)
    other = mu + lam_dom
    return min(lam_dom, other), max(lam_dom, other)   # (most negative, most positive)

In [ ]:
def extreme_eigenvalues(A, **kw):
    """Most positive and most negative eigenvalues (real spectrum assumed)."""
    A = np.asarray(A, dtype=float)
    lam_dom, _ = power_method(A, **kw)
    # Shift so every eigenvalue has the same sign; the dominant one of the
    # shifted matrix is then the opposite end of the spectrum.
    mu, _ = power_method(A - lam_dom * np.eye(A.shape[0]), **kw)
    other = mu + lam_dom
    return min(lam_dom, other), max(lam_dom, other)   # (most negative, most positive)

In [ ]:
print(power_method(eigenA))
print(extreme_eigenvalues(eigenA))
print(shifted_inverse_iteration(eigenA))
print(shifted_inverse_iteration(eigenA, 4.))

### Problem 4.
$a_7/g=$`-0.01030928` is what we get!

In [ ]:
A = np.array([[1, 0, 0, 0, -7],
     [1, -2, 0, 0, 0],
     [0, 1, -3, 0, 0],
     [0, 1, 0, -4, 0],
     [0, 0, 0.5, 0.5, 1]])
b = np.array([7, 0, 3, 4, 0])

In [ ]:
print(np.linalg.solve(A, b))

### Credits
The scipy and numpy documentation, the wikipedia article for the Stefan-Boltzmann law, the wikipedia article for power iteration, Gabe RT, and Claude.